# Analisis de huelgas 1993-2024

Este notebook lee la base maestra homologada, resume cobertura, grafica cruces posibles y verifica la consistencia del modulo de calificacion contra la fuente.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if (ROOT / 'bases').exists() and (ROOT / 'notebooks').exists():
    project_root = ROOT
else:
    project_root = ROOT.parent
BASE = project_root / 'bases' / 'maestra'
GRAPH = BASE / 'graficos'

master = pd.read_csv(BASE / 'huelgas_modulos_maestra_1993_2024.csv')
coverage = pd.read_csv(BASE / 'cobertura_modulos_1993_2024.csv')
legalidad = pd.read_csv(BASE / 'huelgas_legalidad_1996_2024_largo.csv')
legalidad_resumen = pd.read_csv(BASE / 'huelgas_legalidad_1996_2024_resumen_anual.csv')
verif = pd.read_csv(BASE / 'verificacion_calificacion_1993_2024_largo.csv')
verif_resumen = pd.read_csv(BASE / 'verificacion_calificacion_1993_2024_resumen.csv')
anio_sector = pd.read_csv(BASE / 'cruce_anio_sector_largo.csv')
anio_territorio = pd.read_csv(BASE / 'cruce_anio_territorio_regional_largo.csv')


## Cobertura y estructura

La auditoria global y la cobertura por modulo ya vienen consolidadas en la carpeta `bases/maestra`.


In [ ]:
coverage.head(15)


## Legalidad por año

Este es el cruce observable directamente con la fuente. No existe un cruce directo `legalidad x sector` ni `legalidad x territorio` en los anuarios usados para la base principal.


In [ ]:
legalidad_resumen.head()


In [ ]:
for metric, title in [
    ('huelgas', 'Legalidad por año: huelgas'),
    ('trabajadores_comprendidos', 'Legalidad por año: trabajadores comprendidos'),
    ('horas_hombre_perdidas', 'Legalidad por año: horas-hombre perdidas'),
]:
    proc = f'{metric}_procedente'
    ilegal = f'{metric}_ilegal'
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.bar(legalidad_resumen['anio'], legalidad_resumen[proc], label='procedente')
    ax.bar(legalidad_resumen['anio'], legalidad_resumen[ilegal], bottom=legalidad_resumen[proc], label='ilegal')
    ax.set_title(title)
    ax.legend()
    plt.show()


## Cruce año x sector


In [ ]:
pivot_sector = pd.read_csv(BASE / 'cruce_anio_sector_huelgas_pivot.csv').set_index('anio')
pivot_sector.head()


In [ ]:
heat = pivot_sector.copy()
top_cols = heat.sum(axis=0).sort_values(ascending=False).head(12).index
heat = heat[top_cols]
fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(heat.values, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns, rotation=90)
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index.astype(str))
ax.set_title('Año x sector (huelgas, top 12 sectores)')
fig.colorbar(im, ax=ax, label='Huelgas')
plt.show()


## Cruce año x territorio

Se usa solo `nivel_territorial = regional` para evitar duplicar con zonas.


In [ ]:
pivot_territorio = pd.read_csv(BASE / 'cruce_anio_territorio_regional_huelgas_pivot.csv').set_index('anio')
pivot_territorio.head()


In [ ]:
heat = pivot_territorio.copy()
top_cols = heat.sum(axis=0).sort_values(ascending=False).head(12).index
heat = heat[top_cols]
fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(heat.values, aspect='auto', cmap='YlGnBu')
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns, rotation=90)
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index.astype(str))
ax.set_title('Año x territorio regional (huelgas, top 12 territorios)')
fig.colorbar(im, ax=ax, label='Huelgas')
plt.show()


## Sector x territorio

Ese cruce **no forma parte de la base maestra principal** porque requiere el cuadro cruzado `actividad x territorio` de la base complementaria. La nota metodologica consolidada esta en `bases/maestra/nota_sector_territorio.md`.


## Verificación del módulo de calificación

Aquí se comprueba, para cada año con módulo disponible, que `procedente + ilegal` coincide con el `TOTAL` de la hoja fuente en las tres métricas.


In [ ]:
verif_resumen
